In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:
a = 5
b = 5
res = 0.1
scale = 1000


In [ ]:
ipu, m, marker = pattern_generator_using_gmsh.get_square_pillow(a, b, avg_len_boundary = res, avg_len_embeddings = res, scale = scale)


In [ ]:
ipu.sheet.thickness *= scale / 5

In [ ]:
ipu.sheet.thickness

In [ ]:
visualization.plot_2d_mesh(m, pointList = marker, width = 5, height = 5)

In [ ]:
viewer = TriMeshViewer(ipu.sheet, width=768, height=640)


In [ ]:
viewer.showWireframe(False)

In [ ]:
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.1


In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
from matplotlib import cm

In [ ]:
max_stress = 0.0114

In [ ]:
strains = utils.getStrains(ipu.sheet)[:, 0]
strainField = vis.fields.ScalarField(ipu.sheet, strains, colormap = cm.viridis, vmin= 0, vmax =max_stress)

In [ ]:
viewer.update(scalarField=strainField)

In [ ]:
def cb(i):
    strains = utils.getStrains(ipu.sheet)[:, 0]
    strainField = vis.fields.ScalarField(ipu.sheet, strains, colormap = cm.viridis, vmin= 0, vmax = max_stress)
    viewer.update(scalarField=strainField)

In [ ]:
ipu.visualizationTilePower = 0

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-7
print(opts.factorizer)

cr = inflation.inflation_newton(ipu.sheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)
strains = utils.getStrains(ipu.sheet)[:, 0]
strainField = vis.fields.ScalarField(ipu.sheet, strains, colormap = cm.viridis, vmin= 0, vmax = max_stress)
viewer.update(scalarField=strainField)
benchmark.report()

In [ ]:
viewer.saveColorizedObj("square_pillow_{}.obj".format(scale))

In [ ]:
# rescale obj

with open ("square_pillow_{}.obj".format(scale), 'r') as f:
    with open("rescaled_square_pillow_{}.obj".format(scale), 'w') as g:
        content = f.readlines()
        for line in content:
            if 'v ' in line:
                point = np.array([float(x) for x in line.strip().split()[1:]])
                point[:3] *= 5 / scale
                if len(point) == 6:
                    g.write('v {} {} {} {} {} {}\n'.format(*point))
                else:
                    print(line)
            else:
                g.write(line)

In [ ]:
ipu.energy(inflation.InflatableSheet.EnergyType.Elastic), ipu.energy(inflation.InflatableSheet.EnergyType.Pressure)

In [ ]:
ipu.energy(inflation.InflatableSheet.EnergyType.Elastic), ipu.energy(inflation.InflatableSheet.EnergyType.Pressure)

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(ipu.sheet)[:, 0], bins=1000);
plt.xlim(0, 0.013);

In [ ]:
max(utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(ipu.sheet)[:, 0], bins=1000);
plt.xlim(0, 0.013);

In [ ]:
max(utils.getStrains(ipu.sheet)[:, 0])